In [7]:
import os
import glob
import pandas as pd
import numpy as np
import nibabel as nib
from nilearn.glm.first_level import FirstLevelModel
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.exceptions import NotFittedError

# --- 1. (필수) 경로 설정 ---
# 
# ‼️ 중요: 이 경로들을 실제 환경에 맞게 수정해주세요.
# 
# (Windows 경로 오류 방지를 위해 문자열 앞에 r을 붙였습니다.)

# 5명의 피험자 폴더(106016, 108828...)가 들어있는 최상위 폴더
HCP_DATA_ROOT = r"C:/BRAINEEWHA/HCP_Data"

# 모든 피험자의 행동 데이터가 포함된 *하나*의 CSV 파일
BEHAVIORAL_CSV_PATH = r"C:/BRAINEEWHA/HCP_YA_subjects.csv"

# 처리된 3D Z-Map을 저장할 폴더 (자동으로 생성됩니다)
ZMAP_OUTPUT_DIR = r"C:/BRAINEEWHA/processed_zmaps"

# 처리할 피험자 ID 리스트
SUBJECT_LIST = ['103111', '103414', '103818', '105014', '105115']

In [8]:
COL_SUBJECT_ID = 'Subject' # CSV의 피험자 ID 컬럼
COL_Y_TARGET = 'WM_Task_Acc' # 예측 대상 (Y값)

COL_CATEGORICAL = [
    'Gender',  # 범주형 특징 1
    'Age'      # 범주형 특징 2 (예: '31-35')
]

COL_NUMERICAL = [
    'Flanker_AgeAdj',   # Flanker Inhibitory Control Score
    'PMAT24_A_CR',      # Penn Progressive Matrices Score
    'ProcSpeed_AgeAdj', # Pattern Comparison Processing Speed Score
    'ListSort_AgeAdj'   # List Sorting Working Memory Score
] # 수치형 특징 (X_behav)

In [9]:
# --- 2. 전처리기 및 저장소 준비 ---

# Z-Map 저장 폴더 생성
os.makedirs(ZMAP_OUTPUT_DIR, exist_ok=True)
print(f"출력 폴더 확인: {ZMAP_OUTPUT_DIR}")

# 행동 데이터 로드
try:
    behavioral_df_all = pd.read_csv(BEHAVIORAL_CSV_PATH)
    print(f"전체 행동 데이터 로드 완료: {behavioral_df_all.shape}")
except FileNotFoundError:
    print(f"‼️ 오류: 행동 데이터 CSV 파일을 찾을 수 없습니다. BEHAVIORAL_CSV_PATH 경로를 확인하세요.")
    exit()

# X_behav 전처리 파이프라인 정의
# (모든 피험자가 동일한 기준으로 스케일링되도록 루프 *밖에서* 정의)
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), COL_NUMERICAL),
        ('cat', OneHotEncoder(handle_unknown='ignore'), COL_CATEGORICAL)
    ],
    remainder='drop'
)

# 전처리기를 *전체* 데이터로 학습(fit)
try:
    print("행동 데이터 전처리기(Scaler/Encoder) 학습 중...")
    preprocessor.fit(behavioral_df_all[COL_NUMERICAL + COL_CATEGORICAL])
except KeyError as e:
    print(f"‼️ 오류: CSV 파일에서 컬럼을 찾을 수 없습니다: {e}")
    print("COL_Y_TARGET, COL_CATEGORICAL, COL_NUMERICAL 변수를 실제 CSV 컬럼 이름으로 수정하세요.")
    exit()

# 최종 데이터셋을 저장할 리스트
dataset_records = []


출력 폴더 확인: C:/BRAINEEWHA/processed_zmaps
전체 행동 데이터 로드 완료: (5, 10)
행동 데이터 전처리기(Scaler/Encoder) 학습 중...


In [10]:
# --- 3. 피험자별 루프 시작 ---

for subject_id in SUBJECT_LIST:
    print(f"\n--- 🔷 처리 시작: 피험자 {subject_id} ---")
    
    # ‼️ fMRI 처리 실패 시 행동 데이터까지 모두 건너뛰도록 try 하나로 통합
    try:
        # --- 3.1. fMRI GLM 처리 ---
        subject_dir = os.path.join(HCP_DATA_ROOT, subject_id)

        # ✅ .nii 파일을 찾도록 패턴 수정
        # 예: tfMRI_WM_LR_hp0_clean_rclean_tclean.nii
        fmri_search_pattern = os.path.join(
            subject_dir,
            '**',
            'tfMRI_WM_LR',
            'tfMRI_WM_LR_*hp0_clean*.nii'
        )
        fmri_files = glob.glob(fmri_search_pattern, recursive=True)
        print(f"[DEBUG] fMRI 검색 패턴: {fmri_search_pattern}")
        print(f"[DEBUG] 찾은 fMRI 파일 개수: {len(fmri_files)}")
        
        if not fmri_files:
            raise FileNotFoundError(
                f"fMRI 파일(.nii)을 찾을 수 없습니다. 검색 패턴: {fmri_search_pattern}"
            )
        
        fmri_path = fmri_files[0]
        print(f"{subject_id}: fMRI 파일 찾음: {fmri_path}")
        
        # --- EVs 폴더 경로 재귀 검색 ---
        events_dir_search = os.path.join(subject_dir, '**', 'tfMRI_WM_LR', 'EVs')
        events_dirs = glob.glob(events_dir_search, recursive=True)
        print(f"[DEBUG] EVs 검색 패턴: {events_dir_search}")
        print(f"[DEBUG] 찾은 EVs 폴더 개수: {len(events_dirs)}")
        
        if not events_dirs:
            raise FileNotFoundError(
                f"EVs 폴더를 찾을 수 없습니다. 검색 패턴: {events_dir_search}"
            )
            
        events_dir = events_dirs[0]
        print(f"{subject_id}: EVs 폴더 찾음: {events_dir}")

        # --- fMRI 이미지 로드 (.nii) ---
        fmri_img = nib.load(fmri_path)
        print(f"로드된 fMRI 데이터 형태: {fmri_img.shape}")

        # --- 3.2. 여러 .txt 이벤트 파일을 하나의 DataFrame으로 결합 ---
        events_df_list = []
    
        # '0-back' 조건 파일들 (*0bk_*.txt)
        zeroback_files = glob.glob(os.path.join(events_dir, "*0bk_*.txt"))
        print(f"'0-back' 조건 파일 {len(zeroback_files)}개 발견.")
    
        for file_path in zeroback_files:
            temp_df = pd.read_csv(
                file_path,
                sep='\t',
                header=None,
                names=['onset', 'duration', 'intensity']
            )
            temp_df['trial_type'] = 'zeroback'
            events_df_list.append(temp_df)

        # '2-back' 조건 파일들 (*2bk_*.txt)
        twoback_files = glob.glob(os.path.join(events_dir, "*2bk_*.txt"))
        print(f"'2-back' 조건 파일 {len(twoback_files)}개 발견.")

        for file_path in twoback_files:
            temp_df = pd.read_csv(
                file_path,
                sep='\t',
                header=None,
                names=['onset', 'duration', 'intensity']
            )
            temp_df['trial_type'] = 'twoback'
            events_df_list.append(temp_df)
        
        if not events_df_list:
            raise FileNotFoundError(
                f"EVs 폴더에서 *0bk_*.txt / *2bk_*.txt 이벤트 파일을 찾을 수 없습니다: {events_dir}"
            )
        
        events_df = pd.concat(events_df_list)
        events_df = events_df.sort_values(by='onset').reset_index(drop=True)

        print("모든 .txt 이벤트를 하나의 DataFrame으로 성공적으로 결합했습니다.")
        print(events_df.head())

        # --- 3.3. GLM 설정 (FirstLevelModel) ---
        fmri_glm = FirstLevelModel(
            t_r=float(fmri_img.header.get_zooms()[3]),
            noise_model='ar1',
            standardize=False,
            hrf_model='spm',
            drift_model='cosine',
            high_pass=0.01
        )

        # --- 3.4. GLM 학습 ---
        print("GLM 모델 학습 중...")
        fmri_glm = fmri_glm.fit(fmri_img, events=events_df)

        # --- 3.5. 대조 정의 ---
        print("대조(Contrast) 정의: [twoback - zeroback]")

        # --- 3.6. z-map 계산 ---
        z_map = fmri_glm.compute_contrast(
            'twoback - zeroback',
            output_type='z_score'
        )
        print(f"생성된 3D Z-Map 형태: {z_map.shape}")
    
        # --- 3.8. 3D 맵 .nii로 저장 ---
        output_zmap_path = os.path.join(
            ZMAP_OUTPUT_DIR,
            f"{subject_id}_wm_zmap.nii"
        )
        z_map.to_filename(output_zmap_path)
        print(f"3D Z-Map 저장 완료: {output_zmap_path}")

        # --- 3.2. 행동 데이터 처리 ---
        subject_id_int = int(subject_id)
        subject_behavior_row = behavioral_df_all[
            behavioral_df_all[COL_SUBJECT_ID] == subject_id_int
        ]
        
        if subject_behavior_row.empty:
            print(f"경고: CSV에서 {subject_id}의 행동 데이터를 찾지 못했습니다. 건너뜁니다.")
            continue
            
        y_target = subject_behavior_row[COL_Y_TARGET].values[0]
        X_behav_processed = preprocessor.transform(
            subject_behavior_row[COL_NUMERICAL + COL_CATEGORICAL]
        )
        X_behav_vector = X_behav_processed.flatten()
        
        print(f"{subject_id}: 행동 데이터 처리 완료.")
        
        # --- 3.3. 최종 데이터셋에 추가 ---
        dataset_records.append({
            'subject_id': subject_id,
            'zmap_path': output_zmap_path,          # CNN 입력 (X1)
            'behavioral_features': X_behav_vector,  # MLP 입력 (X2)
            'wm_accuracy': y_target                 # 예측 대상 (y)
        })

    except FileNotFoundError as e:
        print(f"‼️ 오류 (파일 없음): {subject_id} 처리 실패. {e}")
    except NotFittedError as e:
        print(f"‼️ 오류 (전처리기): {subject_id} 처리 실패. {e}")
    except Exception as e:
        print(f"‼️ 오류 (알 수 없음): {subject_id} 처리 실패. {e}")



--- 🔷 처리 시작: 피험자 103111 ---
[DEBUG] fMRI 검색 패턴: C:/BRAINEEWHA/HCP_Data\103111\**\tfMRI_WM_LR\tfMRI_WM_LR_*hp0_clean*.nii
[DEBUG] 찾은 fMRI 파일 개수: 2
103111: fMRI 파일 찾음: C:/BRAINEEWHA/HCP_Data\103111\Task3TRecommended\103111\MNINonLinear\Results\tfMRI_WM_LR\tfMRI_WM_LR_Atlas_MSMAll_hp0_clean_rclean_tclean.dtseries.nii
[DEBUG] EVs 검색 패턴: C:/BRAINEEWHA/HCP_Data\103111\**\tfMRI_WM_LR\EVs
[DEBUG] 찾은 EVs 폴더 개수: 1
103111: EVs 폴더 찾음: C:/BRAINEEWHA/HCP_Data\103111\Task3TRecommended\103111\MNINonLinear\Results\tfMRI_WM_LR\EVs
로드된 fMRI 데이터 형태: (405, 91282)
'0-back' 조건 파일 7개 발견.
'2-back' 조건 파일 7개 발견.
모든 .txt 이벤트를 하나의 DataFrame으로 성공적으로 결합했습니다.
    onset  duration intensity trial_type
0   7.997      27.5         1    twoback
1  10.569       2.5         1    twoback
2  13.128       2.5         1    twoback
3  15.687       2.5         1    twoback
4  18.259       2.5         1    twoback
‼️ 오류 (알 수 없음): 103111 처리 실패. 'Cifti2Header' object has no attribute 'get_zooms'

--- 🔷 처리 시작: 피험자 103414 ---
[DEBUG]

C:\Users\82109\AppData\Local\Temp\ipykernel_34424\2549649526.py:85: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  events_df = pd.concat(events_df_list)


로드된 fMRI 데이터 형태: (405, 91282)
'0-back' 조건 파일 7개 발견.
'2-back' 조건 파일 7개 발견.
모든 .txt 이벤트를 하나의 DataFrame으로 성공적으로 결합했습니다.
    onset  duration intensity trial_type
0   7.997      27.5         1    twoback
1  10.556       2.5         1    twoback
2  13.115       2.5         1    twoback
3  15.674       2.5         1    twoback
4  18.246       2.5         1    twoback
‼️ 오류 (알 수 없음): 103414 처리 실패. 'Cifti2Header' object has no attribute 'get_zooms'

--- 🔷 처리 시작: 피험자 103818 ---
[DEBUG] fMRI 검색 패턴: C:/BRAINEEWHA/HCP_Data\103818\**\tfMRI_WM_LR\tfMRI_WM_LR_*hp0_clean*.nii
[DEBUG] 찾은 fMRI 파일 개수: 2
103818: fMRI 파일 찾음: C:/BRAINEEWHA/HCP_Data\103818\Task3TRecommended\103818\MNINonLinear\Results\tfMRI_WM_LR\tfMRI_WM_LR_Atlas_MSMAll_hp0_clean_rclean_tclean.dtseries.nii
[DEBUG] EVs 검색 패턴: C:/BRAINEEWHA/HCP_Data\103818\**\tfMRI_WM_LR\EVs
[DEBUG] 찾은 EVs 폴더 개수: 1
103818: EVs 폴더 찾음: C:/BRAINEEWHA/HCP_Data\103818\Task3TRecommended\103818\MNINonLinear\Results\tfMRI_WM_LR\EVs


C:\Users\82109\AppData\Local\Temp\ipykernel_34424\2549649526.py:85: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  events_df = pd.concat(events_df_list)


로드된 fMRI 데이터 형태: (405, 91282)
'0-back' 조건 파일 7개 발견.
'2-back' 조건 파일 7개 발견.
모든 .txt 이벤트를 하나의 DataFrame으로 성공적으로 결합했습니다.
    onset  duration intensity trial_type
0   7.996      27.5         1    twoback
1  10.569       2.5         1    twoback
2  13.128       2.5         1    twoback
3  15.687       2.5         1    twoback
4  18.259       2.5         1    twoback
‼️ 오류 (알 수 없음): 103818 처리 실패. 'Cifti2Header' object has no attribute 'get_zooms'

--- 🔷 처리 시작: 피험자 105014 ---
[DEBUG] fMRI 검색 패턴: C:/BRAINEEWHA/HCP_Data\105014\**\tfMRI_WM_LR\tfMRI_WM_LR_*hp0_clean*.nii
[DEBUG] 찾은 fMRI 파일 개수: 2
105014: fMRI 파일 찾음: C:/BRAINEEWHA/HCP_Data\105014\Task3TRecommended\105014\MNINonLinear\Results\tfMRI_WM_LR\tfMRI_WM_LR_Atlas_MSMAll_hp0_clean_rclean_tclean.dtseries.nii
[DEBUG] EVs 검색 패턴: C:/BRAINEEWHA/HCP_Data\105014\**\tfMRI_WM_LR\EVs
[DEBUG] 찾은 EVs 폴더 개수: 1
105014: EVs 폴더 찾음: C:/BRAINEEWHA/HCP_Data\105014\Task3TRecommended\105014\MNINonLinear\Results\tfMRI_WM_LR\EVs


C:\Users\82109\AppData\Local\Temp\ipykernel_34424\2549649526.py:85: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  events_df = pd.concat(events_df_list)


로드된 fMRI 데이터 형태: (405, 91282)
'0-back' 조건 파일 7개 발견.
'2-back' 조건 파일 7개 발견.
모든 .txt 이벤트를 하나의 DataFrame으로 성공적으로 결합했습니다.
    onset  duration intensity trial_type
0   7.997      27.5         1    twoback
1  10.569       2.5         1    twoback
2  13.128       2.5         1    twoback
3  15.700       2.5         1    twoback
4  18.273       2.5         1    twoback
‼️ 오류 (알 수 없음): 105014 처리 실패. 'Cifti2Header' object has no attribute 'get_zooms'

--- 🔷 처리 시작: 피험자 105115 ---
[DEBUG] fMRI 검색 패턴: C:/BRAINEEWHA/HCP_Data\105115\**\tfMRI_WM_LR\tfMRI_WM_LR_*hp0_clean*.nii
[DEBUG] 찾은 fMRI 파일 개수: 2
105115: fMRI 파일 찾음: C:/BRAINEEWHA/HCP_Data\105115\Task3TRecommended\105115\MNINonLinear\Results\tfMRI_WM_LR\tfMRI_WM_LR_Atlas_MSMAll_hp0_clean_rclean_tclean.dtseries.nii
[DEBUG] EVs 검색 패턴: C:/BRAINEEWHA/HCP_Data\105115\**\tfMRI_WM_LR\EVs
[DEBUG] 찾은 EVs 폴더 개수: 1
105115: EVs 폴더 찾음: C:/BRAINEEWHA/HCP_Data\105115\Task3TRecommended\105115\MNINonLinear\Results\tfMRI_WM_LR\EVs


C:\Users\82109\AppData\Local\Temp\ipykernel_34424\2549649526.py:85: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  events_df = pd.concat(events_df_list)


로드된 fMRI 데이터 형태: (405, 91282)
'0-back' 조건 파일 7개 발견.
'2-back' 조건 파일 7개 발견.
모든 .txt 이벤트를 하나의 DataFrame으로 성공적으로 결합했습니다.
    onset  duration  intensity trial_type
0   7.997      27.5          1    twoback
1  10.569       2.5          1    twoback
2  13.155       2.5          1    twoback
3  15.741       2.5          1    twoback
4  18.313       2.5          1    twoback
‼️ 오류 (알 수 없음): 105115 처리 실패. 'Cifti2Header' object has no attribute 'get_zooms'


In [5]:
# --- 4. 최종 DataFrame 생성 및 저장 ---

print("\n--- ✅ 모든 피험자 처리 완료 ---")

if not dataset_records:
    print("‼️ 처리된 데이터가 없습니다. 경로와 CSV 컬럼 이름을 다시 확인해주세요.")
else:
    # 리스트(딕셔너리 모음)를 DataFrame으로 변환
    final_dataset_df = pd.DataFrame(dataset_records)
    
    # 최종 데이터셋 CSV 파일로 저장
    final_csv_path = os.path.join(ZMAP_OUTPUT_DIR, 'final_multimodal_dataset.csv')
    final_dataset_df.to_csv(final_csv_path, index=False)
    
    print(f"최종 데이터셋이 {final_csv_path} 에 저장되었습니다.")
    print("\n데이터셋 샘플:")
    print(final_dataset_df.head())


--- ✅ 모든 피험자 처리 완료 ---
‼️ 처리된 데이터가 없습니다. 경로와 CSV 컬럼 이름을 다시 확인해주세요.


In [11]:
import os
import numpy as np
import pandas as pd
from sklearn.exceptions import NotFittedError

# z-map이 저장된 폴더 (Colab에서 만든 zmap들 위치)
ZMAP_OUTPUT_DIR = r"C:\BRAINEEWHA\processed_zmaps"
RUN_NAME = "tfMRI_WM_RL"   # 이번에 쓰는 run 이름 (원래 zmap 파일명에 들어간 부분)

dataset_records = []

# --- 3. 피험자별 루프 시작 ---

for subject_id in SUBJECT_LIST:
    print(f"\n--- 🔷 처리 시작: 피험자 {subject_id} ---")
    
    try:
        # -----------------------------
        # 3.1. z-map 파일 경로만 사용
        # -----------------------------
        # 예: C:\BRAINEEWHA\processed_zmaps\103111_tfMRI_WM_RL_zmap.nii.gz
        zmap_path = os.path.join(
            ZMAP_OUTPUT_DIR,
            f"{subject_id}_{RUN_NAME}_zmap.nii.gz"
        )
        
        if not os.path.exists(zmap_path):
            raise FileNotFoundError(
                f"z-map 파일을 찾을 수 없습니다: {zmap_path}"
            )
        
        print(f"{subject_id}: z-map 파일 찾음 → {zmap_path}")
        
        # -----------------------------
        # 3.2. 행동 데이터 처리
        # -----------------------------
        subject_id_int = int(subject_id)
        subject_behavior_row = behavioral_df_all[
            behavioral_df_all[COL_SUBJECT_ID] == subject_id_int
        ]
        
        if subject_behavior_row.empty:
            print(f"경고: CSV에서 {subject_id}의 행동 데이터를 찾지 못했습니다. 건너뜁니다.")
            continue
        
        # 예측할 타깃 (예: wm_accuracy)
        y_target = subject_behavior_row[COL_Y_TARGET].values[0]
        
        # 숫자 + 범주형 컬럼 전처리
        X_behav_processed = preprocessor.transform(
            subject_behavior_row[COL_NUMERICAL + COL_CATEGORICAL]
        )
        X_behav_vector = X_behav_processed.flatten()
        
        print(f"{subject_id}: 행동 데이터 처리 완료. (feature dim = {X_behav_vector.shape[0]})")
        
        # -----------------------------
        # 3.3. 최종 레코드 추가
        # -----------------------------
        dataset_records.append({
            "subject_id": subject_id,
            "run": RUN_NAME,
            "zmap_path": zmap_path,              # CNN 입력 (X1)
            "behavioral_features": X_behav_vector,  # MLP 입력 (X2)
            "wm_accuracy": y_target              # 예측 대상 (y)
        })
    
    except FileNotFoundError as e:
        print(f"‼️ 오류 (파일 없음): {subject_id} 처리 실패. {e}")
    except NotFittedError as e:
        print(f"‼️ 오류 (전처리기): {subject_id} 처리 실패. {e}")
    except Exception as e:
        print(f"‼️ 오류 (알 수 없음): {subject_id} 처리 실패. {e}")



--- 🔷 처리 시작: 피험자 103111 ---
103111: z-map 파일 찾음 → C:\BRAINEEWHA\processed_zmaps\103111_tfMRI_WM_RL_zmap.nii.gz
103111: 행동 데이터 처리 완료. (feature dim = 9)

--- 🔷 처리 시작: 피험자 103414 ---
103414: z-map 파일 찾음 → C:\BRAINEEWHA\processed_zmaps\103414_tfMRI_WM_RL_zmap.nii.gz
103414: 행동 데이터 처리 완료. (feature dim = 9)

--- 🔷 처리 시작: 피험자 103818 ---
103818: z-map 파일 찾음 → C:\BRAINEEWHA\processed_zmaps\103818_tfMRI_WM_RL_zmap.nii.gz
103818: 행동 데이터 처리 완료. (feature dim = 9)

--- 🔷 처리 시작: 피험자 105014 ---
105014: z-map 파일 찾음 → C:\BRAINEEWHA\processed_zmaps\105014_tfMRI_WM_RL_zmap.nii.gz
105014: 행동 데이터 처리 완료. (feature dim = 9)

--- 🔷 처리 시작: 피험자 105115 ---
105115: z-map 파일 찾음 → C:\BRAINEEWHA\processed_zmaps\105115_tfMRI_WM_RL_zmap.nii.gz
105115: 행동 데이터 처리 완료. (feature dim = 9)


In [12]:
# --- 4. 최종 DataFrame 생성 및 저장 ---

print("\n--- ✅ 모든 피험자 처리 완료 ---")

if not dataset_records:
    print("‼️ 처리된 데이터가 없습니다. 경로와 CSV 컬럼 이름을 다시 확인해주세요.")
else:
    # 리스트(딕셔너리 모음)를 DataFrame으로 변환
    final_dataset_df = pd.DataFrame(dataset_records)
    
    # 최종 데이터셋 CSV 파일로 저장
    final_csv_path = os.path.join(ZMAP_OUTPUT_DIR, "final_multimodal_dataset.csv")
    final_dataset_df.to_csv(final_csv_path, index=False)
    
    print(f"최종 데이터셋이 {final_csv_path} 에 저장되었습니다.")
    print("\n데이터셋 샘플:")
    print(final_dataset_df.head())



--- ✅ 모든 피험자 처리 완료 ---
최종 데이터셋이 C:\BRAINEEWHA\processed_zmaps\final_multimodal_dataset.csv 에 저장되었습니다.

데이터셋 샘플:
  subject_id          run                                          zmap_path  \
0     103111  tfMRI_WM_RL  C:\BRAINEEWHA\processed_zmaps\103111_tfMRI_WM_...   
1     103414  tfMRI_WM_RL  C:\BRAINEEWHA\processed_zmaps\103414_tfMRI_WM_...   
2     103818  tfMRI_WM_RL  C:\BRAINEEWHA\processed_zmaps\103818_tfMRI_WM_...   
3     105014  tfMRI_WM_RL  C:\BRAINEEWHA\processed_zmaps\105014_tfMRI_WM_...   
4     105115  tfMRI_WM_RL  C:\BRAINEEWHA\processed_zmaps\105115_tfMRI_WM_...   

                                 behavioral_features  wm_accuracy  
0  [0.6175005929352042, 0.6620847108818949, -0.43...       92.404  
1  [-0.3092623201765024, -0.9931270663228409, 0.8...       91.679  
2  [-0.04300999154772355, -1.4069300106240248, 0....       85.604  
3  [-1.6302834891423579, 0.6620847108818949, -1.7...       94.000  
4  [1.365055207931387, 1.075887655183079, 0.49443...       89.979 